# Exploring `warpSPHIntegrators`' nonlinear solvers on the implicit wave equation

This notebook is a hands-on reference for the `NonlinearSolver` interface
`warpSPHIntegrators` ships as of **JFNK_PLAN.md Phase A (2026-08-24)** —
`FixedPointSolver` (Picard) and the new `JFNKSolver` (Jacobian-free
Newton-Krylov) — explored on `warpSPH`'s implicit scalar wave equation case,
the smallest problem in this codebase where an implicit solve is genuinely
useful (`warpSPH/tests/test_implicitWaveEquation.py`).

**What you'll find here:**

1. The wave equation problem and its state/system representation.
2. A from-scratch conjugate-gradient reference solve (ground truth).
3. Full documentation of `FixedPointSolver` and `JFNKSolver`'s constructor
   and `solve(...)` options.
4. **Experiment 1** — a `dt` sweep: where Picard iteration diverges and JFNK
   stays bounded.
5. **Experiment 2** — the four registered DIRK tableaus, each driven by
   JFNK, checked for standing-wave convergence with resolution.
6. **Experiment 3** — the finite-difference matvec vs. the exact forward-mode
   JVP matvec: agreement and the concrete Krylov-iteration payoff.
7. **Experiment 4** — a solver-knob playground (`tol`, `max_iterations`) and
   what they mean in float32.
8. A quick-reference cheat sheet for the whole interface.

Every code cell here was verified against the actual installed
`warpSPHIntegrators`/`warpSPH`/`warpSPHCore` before this notebook was
written — the numbers you see when you run it should look like the ones
quoted in the markdown, modulo the usual float32/GPU nondeterminism in the
last couple of digits.

**Requires**: a CUDA GPU (or falls back to CPU, slower), the `warp` conda
environment (`warpSPHCore`, `warpSPHIntegrators`, `warpSPH` all installed
editable), and `matplotlib`/`ipympl` for the interactive plots.

In [ ]:
%matplotlib widget
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

import warp as wp
wp.init()

DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
DTYPE = torch.float32
print(f"device={DEVICE}, dtype={DTYPE}")

plt.rcParams['figure.figsize'] = (7, 4.5)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## 1. The wave equation problem

`warpSPH.schemes.waveEquation.f_wave_equation` solves the scalar wave
equation as a first-order system in `(u, v)`, with a PML-style absorbing
term folded into the acceleration:

$$
\frac{du}{dt} = v, \qquad
\frac{dv}{dt} = c^2 \,\mathrm{Laplacian}(u) - \text{damping} \cdot v
$$

`Laplacian` is `warpSPHCore`'s SPH Laplacian operator (`LaplacianScheme`,
default `Brookshaw`) — the *only* `warpSPHCore` call this scheme makes, and
one of the six operators with an exact forward-mode JVP (`warpOperationJVP`),
which is exactly what makes this case the natural bridge for validating
JFNK's exact-JVP matvec (`NOTES.md` §3.4's correction; `JFNK_PLAN.md`'s own
rationale for starting here).

The state is `WaveSystemStatev3` (`warpSPH.systems.waveSystem`): the usual
particle constants (`positions`, `supports`, `masses`, ...) plus `u`/`v`
(the `integrated`-tagged pair) and `c`/`damping` (`constant`-tagged). `u`/`v`
are tagged `'position'`/`'velocity'` *purely* so `warpSPHIntegrators`
integrates the pair with its position/velocity machinery — this is not a
physical position or velocity, it's reuse of the integrator's plumbing (see
the module's own docstring).

We'll use a 1-D standing-wave initial condition, $u(x, 0) = \sin(kx)$,
$v(x, 0) = 0$, $k=\pi$, constant $c=1$, zero damping — the same case
`test_implicitWaveEquation.py`'s `_buildStandingWaveSystem` builds, with an
analytic solution $u(x,t) = \sin(kx)\cos(ckt)$ to check against.

In [ ]:
from warpSPH.configurations import WaveEquationConfig, buildConfig
from warpSPH.sample.bySamplingScheme import sampleParticles
from warpSPH.schemes.waveEquation import f_wave_equation
from warpSPH.systems.waveSystem import WaveSystemStatev3, WaveSystemv3
from warpSPH.utils import buildDomainDescription
from warpSPHCore import SupportScheme, buildVerletList

CFL_FACTOR = 0.1  # matches test_implicitWaveEquation.py's own default


def buildStandingWaveSystem(nx=32, dim=1, c=1.0, L=2.0, cflFactor=CFL_FACTOR):
    """A periodic 1-D (or dim-D) standing wave u(x,0)=sin(kx), v(x,0)=0,
    k=pi, wave speed c, zero damping -- the validation case this whole
    notebook explores solvers on.
    """
    domain = buildDomainDescription(l=L, dim=dim, periodic=True, device=DEVICE, dtype=DTYPE)
    config, _integrator = buildConfig(dim=dim, nx=nx, domain=domain, device=DEVICE,
                                      dtype=DTYPE, dx=L / nx, cflFactor=cflFactor)
    k = math.pi
    particles = sampleParticles(nx, config)
    x = particles.positions[:, 0]
    u0 = torch.sin(k * x)
    v0 = torch.zeros_like(u0)
    n = x.shape[0]
    state = WaveSystemStatev3(
        positions=particles.positions, supports=particles.supports, masses=particles.masses,
        densities=particles.densities, kinds=torch.zeros(n, device=DEVICE, dtype=torch.int32),
        materials=torch.zeros(n, device=DEVICE, dtype=torch.int32),
        UIDs=torch.arange(n, device=DEVICE, dtype=torch.int32), UIDcounter=n,
        u=u0, v=v0, c=torch.full_like(u0, c), damping=torch.zeros_like(u0),
    )
    adjacency = buildVerletList(state, config.domain, 1.0, SupportScheme.SuperSymmetric, None)
    system = WaveSystemv3(state=state, adjacency=adjacency, domain=config.domain,
                          t=torch.tensor(0.0, device=DEVICE, dtype=DTYPE))
    return system, config, k


schemeConfig = WaveEquationConfig()
print(schemeConfig)

_probe, _config, _k = buildStandingWaveSystem(nx=64)
plt.figure()
plt.plot(_probe.state.positions[:, 0].cpu(), _probe.state.u.cpu(), '.-')
plt.xlabel('x'); plt.ylabel('u(x, 0)'); plt.title('Standing-wave initial condition (nx=64)')
plt.show()

## 2. A ground-truth reference: hand-rolled conjugate gradients

Backward Euler applied to this system eliminates `v` algebraically (the PDE
is linear) down to one stage equation for `u`:

$$
u^{n+1} - \alpha\,\mathrm{Laplacian}(u^{n+1}) = \text{rhs}, \qquad
\alpha = \frac{dt^2 c^2}{1 + dt\cdot\text{damping}}
$$

`test_implicitWaveEquation.py` solves this *exact* `N`-dimensional equation
with a from-scratch CG (matrix-free, via `warpOperationJVP` directly — no
`warpSPHIntegrators` involved at all). We reuse that reference here as
ground truth: it's solving a genuinely *smaller* linear system than the
generic `2N`-dimensional coupled `(u, v)` DIRK stage system JFNK solves, so
agreement between the two is a real cross-check, not a tautology.

In [ ]:
from typing import Callable, Optional, Tuple
from warpSPHCore import OperationProperties, WarpOperation, warpOperationJVP


def conjugateGradientSolve(matvec: Callable[[torch.Tensor], torch.Tensor], b: torch.Tensor,
                           x0: Optional[torch.Tensor] = None, tol: float = 1e-8,
                           maxiter: Optional[int] = None) -> Tuple[torch.Tensor, int]:
    x = x0.clone() if x0 is not None else torch.zeros_like(b)
    r = b - matvec(x)
    p = r.clone()
    rsOld = torch.dot(r, r)
    atol = tol * max(float(torch.linalg.norm(b)), 1.0)
    if maxiter is None:
        maxiter = b.shape[0] * 10
    for iteration in range(maxiter):
        if torch.sqrt(rsOld) < atol:
            return x, iteration
        Ap = matvec(p)
        alpha = rsOld / torch.dot(p, Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rsNew = torch.dot(r, r)
        if torch.sqrt(rsNew) < atol:
            return x, iteration + 1
        p = r + (rsNew / rsOld) * p
        rsOld = rsNew
    return x, maxiter


def implicitBackwardEulerStep(system, dt, config, schemeConfig, tol=1e-8):
    state = system.state
    props = OperationProperties(
        operation=WarpOperation.Laplacian, kernel=schemeConfig.kernel,
        supportMode=schemeConfig.supportMode, laplacianMode=schemeConfig.laplacianMode,
        gradientMode=schemeConfig.gradientMode,
    )

    def laplacian(p):
        return warpOperationJVP(system.state, props, system.domain, tangentQueryValues=p,
                                adjacency=system.adjacency)

    denom = 1.0 + dt * state.damping
    alpha = dt * dt * state.c ** 2 / denom
    rhs = state.u + dt * state.v / denom

    def matvec(p):
        return p - alpha * laplacian(p)

    uNext, _iters = conjugateGradientSolve(matvec, rhs, x0=state.u, tol=tol)
    vNext = (state.v + dt * state.c ** 2 * laplacian(uNext)) / denom

    from warpSPH.systems.waveSystem import WaveSystemStatev3, WaveSystemv3
    newState = WaveSystemStatev3(
        positions=state.positions, supports=state.supports, masses=state.masses,
        densities=state.densities, kinds=state.kinds, materials=state.materials,
        UIDs=state.UIDs, UIDcounter=state.UIDcounter,
        u=uNext, v=vNext, c=state.c, damping=state.damping,
    )
    return WaveSystemv3(state=newState, adjacency=system.adjacency, domain=system.domain,
                        t=system.t + dt)


# Sanity check: one CG step against the analytic solution.
cgSystem, cgConfig, k = buildStandingWaveSystem(nx=64)
dt_small = CFL_FACTOR * cgSystem.state.supports.max().item() / 1.0
cgResult = implicitBackwardEulerStep(cgSystem, dt_small, cgConfig, schemeConfig)
analytic = torch.sin(k * cgResult.state.positions[:, 0]) * math.cos(1.0 * k * dt_small)
print('CG one-step RMSE vs analytic:',
     torch.sqrt(torch.mean((cgResult.state.u - analytic) ** 2)).item())

## 3. The `NonlinearSolver` interface

Every implicit DIRK stage (`warpSPHIntegrators.dirk.DIRK`) needs to solve a
stage equation `Y = base_state + a_ii*dt*f(Y)` for the unknown `Y`. That's
handed to a `NonlinearSolver`:

```python
class NonlinearSolver(Protocol):
    def solve(self, step, y0, norm=None, **opts) -> SolveResult:
        ...  # drives y0 towards a fixed point of step, i.e. y == step(y)

class SolveResult(NamedTuple):
    y: Any            # the solver's answer
    converged: bool    # what "converged" means depends on the solver -- see below
    iterations: int
```

`step(y)` returns the *next* iterate — everything about `dt`, the tableau
coefficient `a_ii`, and the explicit part of the stage is closed over inside
`step`, so every solver in the ladder shares this one call shape.

### `FixedPointSolver` — Picard iteration (the registry default)

```python
FixedPointSolver(iterations: int = 2)
```

Runs `step` a **fixed** number of times, no matter what. With no `tol=`
passed to `solve(...)`, `converged=True` just means "completed its
schedule" — not that anything was actually verified. Two iterations reach
full *order* for the second-order DIRK tableaus (`NOTES.md` §3.1), but that
says nothing about whether the *residual* is actually small — see
Experiment 1 below. Diverges outside its own stability radius
(`|dt*a_ii*L| < 1`, `L` the local Lipschitz constant/Jacobian norm); more
iterations of a divergent map just diverge *faster*, they never help.

### `JFNKSolver` — Jacobian-free Newton-Krylov (opt-in, `JFNK_PLAN.md` Phase A)

```python
JFNKSolver(
    matvec: str = 'fd',            # 'fd' (finite-difference) or 'jvp' (exact)
    tol: float = 1e-8,             # Newton convergence: norm(Y, step(Y)) < tol
    max_iterations: int = 20,      # Newton *correction* budget
    gmres_tol: float = None,       # defaults to `tol`
    gmres_maxiter: int = None,     # defaults to the state's own flat length
    gmres_restart: int = 30,       # GMRES(m) restart length
    fd_eps: float = None,          # override the Knoll-Keyes FD step
)
```

Every option above is also a `solve(...)` **kwarg**, i.e.
`solver_opts={'tol': 1e-6}` on a DIRK call overrides the constructor default
for that one call, matching `FixedPointSolver`'s own `opts.get(..., self.x)`
pattern.

Each outer (Newton) iteration: evaluate `step(Y)` once, check
`norm(Y, step(Y)) < tol`; if not converged, solve one Newton correction with
GMRES against a matrix-free Jacobian-vector product, and repeat. The number
of `step()` calls is `max_iterations + 1` in the worst case (one extra to
verify/report on the last correction). **Never opt into this by default** —
it's real machinery (flatten/unflatten, GMRES, a matvec choice) for a
problem `FixedPointSolver` already handles well in the non-stiff regime.
Nothing in the scheme registry ever picks it automatically.

**`matvec='fd'`** (default): `Jv ≈ (G(Y+εv) - G(Y))/ε`, works for *any*
`step`, no capability gate. `ε` is the standard Knoll-Keyes scaled step,
`sqrt(eps_machine)*(1+‖y‖)/‖v‖`.

**`matvec='jvp'`**: exact, via `torch.autograd.forward_ad`. Only exact when
`step` is built entirely from `warpSPHCore`'s six JVP-wrapped operators
(Density, Interpolate, Gradient, Divergence, Curl, Laplacian) plus
Covariance — which the wave equation's Laplacian call satisfies. For
anything else, this **raises `NotImplementedError` loudly** rather than
silently dropping the missing term — never caught anywhere in this stack.

`norm=None` (the default when `JFNKSolver.solve` is called directly, e.g.
Experiment 4 below) falls back to a plain relative L2 norm over the
flattened state, `‖a-b‖ / max(‖a‖, 1)`. Driven through
`get_scheme(...)`/`DIRK`, `dirk.py` always supplies its own
Hairer-Wanner-weighted `norm` instead — a *different* scale, so a `tol`
tuned against one convention isn't directly portable to the other (see
Experiment 4).

### The four registered DIRK tableaus

```python
from warpSPHIntegrators import get_scheme
get_scheme('Backward Euler (implicit)')   # order 1, L-stable
get_scheme('Implicit Midpoint')           # order 2, A-stable, symplectic (Picard default drifts -- see NOTES.md)
get_scheme('Trapezoidal (Crank-Nicolson)')# order 2, A-stable (not L)
get_scheme('SDIRK2')                      # order 2, L-stable
```

Calling one runs a full step:
`scheme(system, dt, f_wave_equation, config, schemeConfig, solver=...)` →
`IntegrationResult(state=..., stages=..., ...)`.

In [ ]:
from warpSPHIntegrators.solvers import FixedPointSolver, JFNKSolver
from warpSPHIntegrators.state import get_reference_state
from warpSPHIntegrators import get_scheme


def runStep(system, dt, config, schemeConfig, solver, schemeName='Backward Euler (implicit)'):
    scheme = get_scheme(schemeName)
    return scheme(system, dt, f_wave_equation, config, schemeConfig, solver=solver)


# quick smoke test of every solver on one small step
_sys, _config, _k = buildStandingWaveSystem(nx=32)
dt0 = CFL_FACTOR * _sys.state.supports.max().item()
for name, solver in [
    ('Picard(2)', FixedPointSolver(iterations=2)),
    ('JFNK(fd)', JFNKSolver(matvec='fd')),
    ('JFNK(jvp)', JFNKSolver(matvec='jvp')),
]:
    s, c, _ = buildStandingWaveSystem(nx=32)
    r = runStep(s, dt0, c, schemeConfig, solver)
    print(f'{name:12s} u[:3] = {get_reference_state(r.state).u[:3].tolist()}')

## Experiment 1 — `dt` sweep: where Picard diverges and JFNK doesn't

`FixedPointSolver`'s stability radius is set by `dt*a_ii*L`, `L` the local
Lipschitz constant of `f` — here set by the Laplacian operator's spectral
radius, which scales like $c^2/h^2$. At `nx=32`, `L≈2`, the case's own
CFL-scaled `dt` is only about `6e-3`; we'll sweep `dt` from well below that
up past $10^3\times$ it.

We use `JFNK(jvp)` at a tight tolerance as the "truth" reference for this
sweep (rather than the CG solver above) precisely *because* it's solving
the same `2N`-dimensional system every other solver here solves — a fairer
apples-to-apples comparison than the CG reference's hand-eliminated
`N`-dimensional system. We'll cross-check against CG once at the end.

In [ ]:
def relError(u, uRef):
    return float(torch.linalg.norm(u - uRef) / torch.linalg.norm(uRef))


dts = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
nx = 32
solvers = {
    'Picard(2) [default]': lambda: FixedPointSolver(iterations=2),
    'Picard(20)': lambda: FixedPointSolver(iterations=20),
    'JFNK(fd)': lambda: JFNKSolver(matvec='fd', tol=1e-8, max_iterations=15),
    'JFNK(jvp)': lambda: JFNKSolver(matvec='jvp', tol=1e-8, max_iterations=15),
}

results = {name: [] for name in solvers}
for dt in dts:
    refSys, refConfig, _k = buildStandingWaveSystem(nx=nx)
    refResult = runStep(refSys, dt, refConfig, schemeConfig, JFNKSolver(matvec='jvp', tol=1e-10, max_iterations=20))
    uRef = get_reference_state(refResult.state).u

    for name, factory in solvers.items():
        s, c, _ = buildStandingWaveSystem(nx=nx)
        r = runStep(s, dt, c, schemeConfig, factory())
        u = get_reference_state(r.state).u
        err = relError(u, uRef) if torch.isfinite(u).all() else float('inf')
        results[name].append(err)

fig, ax = plt.subplots()
for name, errs in results.items():
    plotErrs = [e if e > 0 else 1e-20 for e in errs]  # log-safe
    ax.loglog(dts, plotErrs, 'o-', label=name)
ax.set_xlabel('dt'); ax.set_ylabel('relative error vs. JFNK(jvp, tight tol)')
ax.set_title(f'Backward Euler stage solve, nx={nx} -- Picard vs. JFNK across dt')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()

for dt, *rows in zip(dts, *results.values()):
    print(f'dt={dt:7.3f}  ' + '  '.join(f'{n}={v:.2e}' for n, v in zip(results, rows)))

**What to expect**: `Picard(2)`'s error grows roughly linearly with `dt`
(it never diverges *outright* at 2 iterations — it just returns something
increasingly wrong). `Picard(20)` tracks `Picard(2)` at small `dt`, but past
the stability radius it **diverges explosively** with the extra iterations
(easily `1e11`+ by `dt=1`, `inf` — float32 overflow — by `dt≈2-5`): running
a divergent fixed-point map harder makes it *more* wrong, not less. Both
`JFNK` variants stay within a few `1e-6`–`1e-7` of the reference at *every*
`dt` in the sweep — Newton on this exactly-linear problem is exact, modulo
GMRES's own tolerance, regardless of stiffness.

This is `JFNK_PLAN.md` A6's own validation case, generalized into a sweep:
`tests/test_implicitWaveEquation.py::test_picardDivergesWhereJFNKStaysBoundedOnAStiffStep`
checks exactly this at a single `dt=2.0`.

## Experiment 2 — all four DIRK tableaus, checked against the analytic solution

Standing-wave error at a *fixed*, CFL-scaled `dt` is dominated by **spatial**
(SPH) discretization error at this resolution, not temporal truncation
error — so this isn't a clean way to measure a tableau's temporal order (for
that, `warpSPHIntegrators`'s own point-ODE probes in `testing.py`
(`oscillator`/`forced`/`damped`) are the right tool, as `test_dirk.py` uses).
What *this* case is good for is the same check
`test_implicitStandingWaveErrorShrinksWithResolution` already runs for the
CG-based backward Euler path: does the standing wave converge to the
analytic solution as resolution increases, for every tableau, driven by
JFNK instead of the hand-eliminated CG solve?

In [ ]:
def standingWaveError(schemeName, solverFactory, nx=32, dim=1, c=1.0, L=2.0):
    system, config, k = buildStandingWaveSystem(nx, dim, c, L)
    hMax = system.state.supports.max().item()
    dt = CFL_FACTOR * hMax / c
    tEnd = (2 * math.pi / (c * k)) * 0.25
    nSteps = max(1, round(tEnd / dt))
    dtActual = tEnd / nSteps
    for _ in range(nSteps):
        result = runStep(system, dtActual, config, schemeConfig, solverFactory(), schemeName=schemeName)
        system = result.state
    x = system.state.positions[:, 0]
    analytic = torch.sin(k * x) * math.cos(c * k * tEnd)
    return torch.sqrt(torch.mean((system.state.u - analytic) ** 2)).item()


schemeNames = ['Backward Euler (implicit)', 'Implicit Midpoint', 'Trapezoidal (Crank-Nicolson)', 'SDIRK2']
resolutions = (32, 64, 128)

fig, ax = plt.subplots()
for schemeName in schemeNames:
    errs = [standingWaveError(schemeName, lambda: JFNKSolver(matvec='fd', tol=1e-9, max_iterations=15), nx=nx)
           for nx in resolutions]
    ax.loglog(resolutions, errs, 'o-', label=schemeName)
    print(f'{schemeName:32s} {["%.3e" % e for e in errs]}')
ax.set_xlabel('nx (resolution)'); ax.set_ylabel('standing-wave RMSE vs. analytic')
ax.set_title('JFNK(fd) through every registered DIRK tableau')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()

Error shrinks monotonically with resolution for all four tableaus, driven
entirely by `JFNKSolver` — the generic driver reaching the same qualitative
answer as the problem-specific CG path, for every registered implicit
scheme, not just backward Euler.

## Experiment 3 — FD vs. exact-JVP matvec: agreement and the concrete payoff

The two matvecs approximate the *same* linear operator ($J_G$, the Jacobian
of the stage residual $G(Y) = Y - \text{step}(Y)$). We check they agree to
FD's own truncation tolerance, and — the actual reason rung 3 exists at all
— that JVP needs no more (usually fewer) GMRES iterations to converge,
since it has no truncation error and no `ε` to tune.

This works directly with `fd_matvec`/`jvp_matvec`/`gmres` from
`warpSPHIntegrators.solvers.krylov` — the same primitives `JFNKSolver` uses
internally, exposed so you can build custom solve loops (or just inspect
matvec behavior, as here) without going through the full solver.

In [ ]:
from warpSPHIntegrators.state import flatten_integrated
from warpSPHIntegrators.solvers import fd_matvec, gmres, jvp_matvec
from warpSPHIntegrators.state import updateStateEuler
from warpSPHIntegrators.state.lifecycle import updateStep


def backwardEulerStageStepFn(system, dt, config, schemeConfig):
    """The same stage map DIRK builds internally for backward Euler's one
    implicit stage (a_ii=1): Y -> y0 + dt*f(Y). Exposed here so the matvecs
    can be driven directly, outside of JFNKSolver's own Newton loop.
    """
    y0 = system.initializeNewState()
    y0.t = float(system.t) + dt

    def step_fn(Y):
        Y.t = y0.t
        k, r = updateStep(system, Y, dt, f_wave_equation, config, schemeConfig)
        return updateStateEuler(y0, k, dt, copyState=True)

    return y0, step_fn


dts = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
iters_fd, iters_jvp, matvec_diff = [], [], []
torch.manual_seed(0)
for dt in dts:
    system, config, _k = buildStandingWaveSystem(nx=32)
    y0, step_fn = backwardEulerStageStepFn(system, dt, config, schemeConfig)
    y_flat = flatten_integrated(y0)
    G_y = y_flat - flatten_integrated(step_fn(y0))

    mv_fd = fd_matvec(step_fn, y0, y_flat, G_y)
    mv_jvp = jvp_matvec(step_fn, y0)

    v = torch.randn_like(y_flat)
    matvec_diff.append(float(torch.linalg.norm(mv_fd(v) - mv_jvp(v)) / torch.linalg.norm(mv_jvp(v))))

    _delta_fd, n_fd = gmres(mv_fd, -G_y, tol=1e-8, maxiter=y_flat.numel())
    _delta_jvp, n_jvp = gmres(mv_jvp, -G_y, tol=1e-8, maxiter=y_flat.numel())
    iters_fd.append(n_fd)
    iters_jvp.append(n_jvp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.loglog(dts, matvec_diff, 'o-', color=colors[3])
ax1.set_xlabel('dt'); ax1.set_ylabel('relative matvec disagreement (fd vs jvp)')
ax1.set_title("FD's own truncation error vs. dt")
ax1.grid(True, which='both', alpha=0.3)

ax2.semilogx(dts, iters_fd, 'o-', label='fd matvec')
ax2.semilogx(dts, iters_jvp, 's-', label='jvp matvec')
ax2.set_xlabel('dt'); ax2.set_ylabel('GMRES iterations to tol=1e-8')
ax2.set_title('Krylov cost: fd vs. exact jvp')
ax2.legend(); ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

for dt, nf, nj, d in zip(dts, iters_fd, iters_jvp, matvec_diff):
    print(f'dt={dt:6.3f}  gmres(fd)={nf:3d}  gmres(jvp)={nj:3d}  matvec rel.diff={d:.2e}')

Two things to read off this plot: (1) the FD/JVP matvec disagreement grows
with `dt` (FD's own `ε` is tuned to the current iterate/direction, not to
`dt`, so as the operator's scale grows with `dt` the fixed-precision FD
formula loses relative accuracy) — never so large that FD becomes *wrong*
for GMRES's purposes, just less exact, exactly as `JFNK_PLAN.md` A6
anticipated ("agree to FD's own truncation tolerance"). (2) JVP never needs
*more* Krylov iterations than FD to reach the same tolerance, and at some
`dt` values needs meaningfully fewer — the concrete reason to reach for
`matvec='jvp'` once you know your `step` qualifies.

## Experiment 4 — solver knobs: `tol` and `max_iterations` in float32

`JFNKSolver.solve(step, y0)` can be called directly (bypassing DIRK/`norm`
entirely) to see exactly what `SolveResult` reports for a given `tol`/
`max_iterations`. Two things worth knowing before you tune these on a real
problem:

- **`tol` has a float32 floor.** This state is `float32` throughout (the
  whole `warpSPH`/`warpSPHCore` stack defaults to it); asking for a relative
  tolerance tighter than roughly float32's own epsilon (~`1.2e-7`) simply
  can't be satisfied — `SolveResult.converged` will be `False` and
  `iterations` will hit your `max_iterations` cap every time, not because
  anything is wrong, but because the target is below the noise floor of the
  arithmetic itself.
- **`max_iterations` is a correction *budget*, not a call count.** The
  number of `step()` evaluations is `max_iterations + 1` in the
  non-converged case (one more to report on the last correction) — see
  `JFNKSolver`'s own docstring.

In [ ]:
system, config, _k = buildStandingWaveSystem(nx=32)
dt = 0.1
y0, step_fn = backwardEulerStageStepFn(system, dt, config, schemeConfig)

print('--- varying tol (max_iterations=20) ---')
for tol in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]:
    result = JFNKSolver(matvec='fd', tol=tol, max_iterations=20).solve(step_fn, y0)
    print(f'  tol={tol:9.1e}  converged={result.converged!s:5s}  iterations={result.iterations}')

print('--- varying max_iterations (tol=1e-10, unreachable in float32) ---')
for max_it in [1, 2, 5, 20]:
    result = JFNKSolver(matvec='fd', tol=1e-10, max_iterations=max_it).solve(step_fn, y0)
    print(f'  max_iterations={max_it:3d}  converged={result.converged!s:5s}  iterations={result.iterations}')

## Quick reference

| | `FixedPointSolver` | `JFNKSolver` |
|---|---|---|
| Registry default | **Yes**, every DIRK scheme | No — always `solver=JFNKSolver(...)` |
| Cost per stage | Fixed: `iterations` calls to `step` | Variable: up to `max_iterations+1` calls to `step`, each non-final one plus one `gmres` solve |
| Stiff problems | Diverges past `\|dt*a_ii*L\|<1`; more iterations make it *worse* | Designed for this — Newton on a linear residual is exact |
| Differentiable / CUDA-graph-capturable | Yes, by construction (fixed unroll depth) | Not evaluated here — branches on `norm`/`tol` |
| Constructor | `FixedPointSolver(iterations=2)` | `JFNKSolver(matvec='fd', tol=1e-8, max_iterations=20, gmres_tol=None, gmres_maxiter=None, gmres_restart=30, fd_eps=None)` |
| Per-call override | `solver_opts={'iterations': N, 'tol': ..., 'norm': ...}` | `solver_opts={'matvec': ..., 'tol': ..., 'max_iterations': ..., 'gmres_tol': ..., 'gmres_maxiter': ..., 'gmres_restart': ..., 'fd_eps': ...}` |
| `converged` meaning | `True` if `tol=None` (schedule completed) or the tol check passed | `True` only if `norm(Y, step(Y)) < tol` was actually reached |

**Driving a DIRK scheme with a chosen solver:**
```python
from warpSPHIntegrators import get_scheme
from warpSPHIntegrators.solvers import JFNKSolver
scheme = get_scheme('Backward Euler (implicit)')  # or 'Implicit Midpoint' / 'Trapezoidal (Crank-Nicolson)' / 'SDIRK2'
result = scheme(system, dt, f_wave_equation, config, schemeConfig,
                solver=JFNKSolver(matvec='jvp', tol=1e-8))
result.state  # the new WaveSystemv3
```

**Low-level primitives** (`warpSPHIntegrators.state.fields` / `.jfnk`), useful for
custom solve loops or, as in Experiments 3-4 above, for inspecting solver
internals directly:

```python
from warpSPHIntegrators.state.fields import flatten_integrated, unflatten_integrated
from warpSPHIntegrators.solvers.krylov import fd_matvec, jvp_matvec, gmres
```

**Further reading**: `warpSPHIntegrators/JFNK_PLAN.md` (the full phased
plan — Phase A is what this notebook exercises; Phases B-D build on it
towards making WCSPH's acoustic subsystem implicit), `NOTES.md` §3.4 (the
solver ladder this fits into) and §3.6 (the DIRK driver itself),
`warpSPH/tests/test_implicitWaveEquation.py` (the test suite this notebook
mirrors as exploratory code), `warpSPHIntegrators/tests/test_jfnk.py` (unit
tests for the primitives used in Experiments 3-4).